In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, make_column_selector

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
exp_name = 'imu_only_simple'
exp_name = 'fft_v1_baseline' # Named for your experiment
sampling_rate = 100
eg_subject = 'SUBJ_040724'
single_sequence = 'SEQ_065189'
dc_offset_max = 10
phase_1_cols = ['acc_x', 'acc_y', 'acc_z', 'sequence_counter', 'phase', 'behavior', 'orientation', 'subject', 'gesture', 'sequence_id']
log_edges = np.logspace(np.log10(0.5), np.log10(100), num=15)
sequences = ['SEQ_000034', 'SEQ_065478', 'SEQ_065470']
subjects = ['SUBJ_059520']
band_edges = np.arange(0, 101, 10)
pipe_name = 'imu_extractor'
categorical_features = ['orientation', 'subject']
accelerometer_combinations = [
    ['acc_x'], ['acc_y'], ['acc_z'],
    ['acc_x', 'acc_y'], ['acc_x', 'acc_z'], ['acc_y', 'acc_z'],
    ['acc_x', 'acc_y', 'acc_z']
]
subject_cols = ['orientation', 'subject', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
some_subjects = [ 'SUBJ_049223', 'SUBJ_019262', 'SUBJ_053906']

In [ ]:
train_df = raw_train_df.set_index('row_id')
single_subject_df = train_df.loc[train_df['subject'] == eg_subject, :]
single_gesture_df = single_subject_df.loc[single_subject_df['sequence_id'] == single_sequence]

multiple_gesture_df = train_df.loc[train_df['subject'].isin(subjects), phase_1_cols]
subject_df = train_df[train_df['subject'].isin(some_subjects)]

In [ ]:
test_df = utils.ImuExtractor(subject_df=train_demo_df).transform(multiple_gesture_df)
keywords = ['orientation', 'subject', 'sequence_type', 'sequence_id', 'behavior', 'phase', 'gesture']
pattern = '|'.join(keywords)

relevant_cols = test_df.columns[~test_df.columns.str.contains(pattern)]
corr_matrix = test_df[relevant_cols].corr()
plt.figure(figsize=(15, 15))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
num_pattern = 'acc'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']

preprocessor = ColumnTransformer(
    transformers=[
        ('feature_num_cols', StandardScaler(), make_column_selector(pattern=num_pattern)),
        ('subject_num_cols', StandardScaler(), suspect_cols),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['orientation', 'subject']),
        ('normal_cols', 'passthrough', ['adult_child', 'sex', 'handedness'])
    ],
    remainder='drop' 
).set_output(transform="pandas")

pipeline = Pipeline([
    (pipe_name, utils.ImuExtractor(subject_df=train_demo_df)),
    ('preprocessor', preprocessor),
    ('classifier', utils.ManyToOneWrapper(RandomForestClassifier()))
])

param_grid = {
    f'{pipe_name}__imu_sensor_list': [['acc_x']],
    f'{pipe_name}__domain': ['acceleration'],
    f'{pipe_name}__dc_offset': [2.0],
    f'{pipe_name}__band_edges': [None]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=GroupKFold(n_splits=3),
    verbose=1,
    n_jobs=1
)

y = subject_df[['sequence_id', 'gesture']]

grid_search.fit(subject_df, y, groups=subject_df['sequence_id'])

results_df = pd.DataFrame(grid_search.cv_results_)
results_df